## 六、综合案例：工单文本分类基线

### 6.1 任务说明

根据工单文本，将问题分为以下四类：

- `物流问题`
- `退款问题`
- `账户问题`
- `商品问题`

| 字段          | 含义           |
| ----------- | ------------ |
| `ticket_id` | 工单编号，只用于定位样本 |
| `text`      | 原始工单文本       |
| `category`  | 人工标注的问题类别    |

### 6.2 读取、检查并清洗数据

In [1]:
import re
import pandas as pd
from pathlib import Path

In [2]:
def clean_text(text: str) -> str:
    """删除简单 HTML 标签并合并多余空白。"""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [3]:
# 在 4.AI全景认知 目录中运行 Notebook
data_path = Path('data/05_tickets.csv')
tickets = pd.read_csv(data_path)

print('数据形状: ', tickets.shape)
print('缺失值: \n', tickets.isna().sum())
print('重复文本数量: ', tickets['text'].duplicated().sum())
print('类别数量: \n', tickets['category'].value_counts())

数据形状:  (120, 3)
缺失值: 
 ticket_id    0
text         0
category     0
dtype: int64
重复文本数量:  0
类别数量: 
 category
物流问题    30
退款问题    30
账户问题    30
商品问题    30
Name: count, dtype: int64


In [4]:
# 工单编号应唯一，标签只能属于规定的四个类别
expected_categories = {
    "物流问题",
    "退款问题",
    "账户问题",
    "商品问题"
}

assert tickets['ticket_id'].is_unique
assert set(tickets['category']) == expected_categories

# 轻量清洗不需要从整批数据学习统计规则，可以在划分前执行
tickets['cleaned_text'] = tickets['text'].map(clean_text)
print(tickets[['text', 'cleaned_text']].head())

                    text cleaned_text
0      <p>订单  一直没有发货</p>    订单 一直没有发货
1  <div>申请退款  不想要了</div>    申请退款 不想要了
2        <p>账号  无法登录</p>      账号 无法登录
3     <p>收到的商品  已经破损</p>   收到的商品 已经破损
4               快递三天没有更新     快递三天没有更新


这里保留原始 `text` 用于查看错误样本，模型实际使用 `cleaned_text`。

### 6.3 划分训练集、验证集和测试集

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X = tickets['cleaned_text']
y = tickets['category']

# 第一次划分：60% 训练集，40% 暂存
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    # 保持四个类别的比例接近
    stratify=y
)

In [7]:
# 第二次划分：把暂存数据平均分成验证集和测试集
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [8]:
print(
    f'训练集: {len(X_train)}条,'
    f'验证集: {len(X_valid)}条,'
    f'测试集: {len(X_test)}条'
)

训练集: 72条,验证集: 24条,测试集: 24条


### 6.4 建立 Pipeline 并在验证集上评估

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

In [10]:
# Pipeline 按顺序完成文本向量化和模型训练
text_classifier = Pipeline(
    steps=[
        (
            # 步骤名称可以自定义，用于识别和访问该步骤
            'tfidf',
            TfidfVectorizer(
                # 使用字符特征，不依赖中文分词库
                analyzer='char',
                # 同时使用单个字符和相邻两个字符
                ngram_range=(1, 2),
            )
        ),
        (
            'model',
            LogisticRegression(
                # 增加最大迭代次数，避免模型尚未收敛
                max_iter=1000,
                # 固定可能涉及的随机过程
                random_state=42
            )
        )
    ]
)

In [11]:
# Pipeline 只使用训练集建立词表并训练模型
text_classifier.fit(X_train, y_train)
# 验证集用于评估当前方案和分析错误
valid_predictions = text_classifier.predict(X_valid)

In [12]:
# 输出各类别的 Precision、Recall 和 F1
print(
    classification_report(
        y_valid,
        valid_predictions,
        #保留3位小数
        digits=3,
        #没有预测到某个类别时返回 0，不产生警告
        zero_division=0
    )
)

              precision    recall  f1-score   support

        商品问题      0.750     1.000     0.857         6
        物流问题      1.000     0.833     0.909         6
        账户问题      1.000     0.833     0.909         6
        退款问题      1.000     1.000     1.000         6

    accuracy                          0.917        24
   macro avg      0.938     0.917     0.919        24
weighted avg      0.938     0.917     0.919        24



In [13]:
# 根据验证集索引找回工单编号和原始文本
validation_results = tickets.loc[
    X_valid.index,
    ['ticket_id', 'text', 'category']
].copy()

validation_results['predicted_category'] = valid_predictions

validation_errors = validation_results[
    validation_results["category"]
    != validation_results["predicted_category"]
]

print('验证集错误样本: \n', validation_errors.to_string(index=False))

验证集错误样本: 
 ticket_id       text category predicted_category
 NLP-0113 同城配送超过预计时间     物流问题               商品问题
 NLP-0071  绑定邮箱已经停用了     账户问题               商品问题


### 6.6 最终测试与新工单预测

In [14]:
# 测试集只用于最后一次评估
test_predictions = text_classifier.predict(X_test)

print("测试集分类报告：")
print(
    classification_report(
        y_test,
        test_predictions,
        digits=3,
        zero_division=0,
    )
)

# 使用训练完成的 Pipeline 预测新工单
new_tickets = [
    "我的包裹已经好几天没有动了",
    "账号验证码一直收不到",
    "收到的耳机少了一只",
]

new_predictions = text_classifier.predict(new_tickets)

for text, label in zip(new_tickets, new_predictions):
    print(f"{text} -> {label}")

测试集分类报告：
              precision    recall  f1-score   support

        商品问题      1.000     0.833     0.909         6
        物流问题      0.800     0.667     0.727         6
        账户问题      0.833     0.833     0.833         6
        退款问题      0.750     1.000     0.857         6

    accuracy                          0.833        24
   macro avg      0.846     0.833     0.832        24
weighted avg      0.846     0.833     0.832        24

我的包裹已经好几天没有动了 -> 物流问题
账号验证码一直收不到 -> 账户问题
收到的耳机少了一只 -> 商品问题


## 九、课后作业

1. **基础题**：准备 5 条包含 HTML 标签、多余空格或标点的客服文本，使用 `clean_text()` 清洗；再使用 jieba 分词，并添加至少 2 个业务词观察分词变化。

In [15]:
import re
import jieba

In [16]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"([，。！？；：、,.!?;:])\1+", r"\1", text)
    return text.strip()

In [17]:
raw_texts = [
    "<p>我的订单  一直  没发货！！！！！</p>",
    "退款申请<br/>已经提交了，，，什么时候能到账？？？",
    "  <div>快递太慢了    都三天了还没到</div>  ",
    "商品质量有问题！！！   <b>要求退货</b>？？",
    "客服  态度  很好，但是  问题  没解决。。。。",
]

In [18]:
for i, t in enumerate(raw_texts, 1):
    print(f'{i}.{t!r}')

1.'<p>我的订单  一直  没发货！！！！！</p>'
2.'退款申请<br/>已经提交了，，，什么时候能到账？？？'
3.'  <div>快递太慢了    都三天了还没到</div>  '
4.'商品质量有问题！！！   <b>要求退货</b>？？'
5.'客服  态度  很好，但是  问题  没解决。。。。'


In [19]:
# 清洗
cleaned_texts = [clean_text(t) for t in raw_texts]

for i, t in enumerate(cleaned_texts, 1):
    print(f'{i}.{t}')

1.我的订单 一直 没发货！
2.退款申请 已经提交了，什么时候能到账？
3.快递太慢了 都三天了还没到
4.商品质量有问题！ 要求退货 ？
5.客服 态度 很好，但是 问题 没解决。


In [20]:
# 添加业务词前，先分词
before = [jieba.lcut(t) for t in cleaned_texts]

for i, tokens in enumerate(before, 1):
    print(f'{i}.{tokens}')

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.636 seconds.
Prefix dict has been built successfully.


1.['我', '的', '订单', ' ', '一直', ' ', '没', '发货', '！']
2.['退款', '申请', ' ', '已经', '提交', '了', '，', '什么', '时候', '能', '到', '账', '？']
3.['快递', '太慢', '了', ' ', '都', '三天', '了', '还', '没到']
4.['商品质量', '有', '问题', '！', ' ', '要求', '退货', ' ', '？']
5.['客服', ' ', '态度', ' ', '很', '好', '，', '但是', ' ', '问题', ' ', '没', '解决', '。']


In [21]:
# 添加至少 2 个业务词
jieba.add_word("没发货")
jieba.add_word("到账")
jieba.add_word("退货")

In [22]:
after = [jieba.lcut(t) for t in cleaned_texts]

for i, tokens in enumerate(after, 1):
    print(f'{i}.{tokens}')

1.['我', '的', '订单', ' ', '一直', ' ', '没发货', '！']
2.['退款', '申请', ' ', '已经', '提交', '了', '，', '什么', '时候', '能', '到账', '？']
3.['快递', '太慢', '了', ' ', '都', '三天', '了', '还', '没到']
4.['商品质量', '有', '问题', '！', ' ', '要求', '退货', ' ', '？']
5.['客服', ' ', '态度', ' ', '很', '好', '，', '但是', ' ', '问题', ' ', '没', '解决', '。']


In [23]:
for i, (b, a) in enumerate(zip(before, after), 1):
    if b != a:
        print(f"{i}. 前: {b}")
        print(f"   后: {a}")

1. 前: ['我', '的', '订单', ' ', '一直', ' ', '没', '发货', '！']
   后: ['我', '的', '订单', ' ', '一直', ' ', '没发货', '！']
2. 前: ['退款', '申请', ' ', '已经', '提交', '了', '，', '什么', '时候', '能', '到', '账', '？']
   后: ['退款', '申请', ' ', '已经', '提交', '了', '，', '什么', '时候', '能', '到账', '？']


2. **综合题**：保持验证集和测试集不变，只为训练集中的每个类别补充 5 条文本。重新训练模型，对比修改前后的验证集指标和错误样本，再选择最终模型并只在测试集上评估一次。

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [25]:
data_path = Path('data/05_tickets.csv')
tickets = pd.read_csv(data_path)

In [26]:
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

tickets['text_clean'] = tickets['text'].apply(clean_text)
print(tickets['category'].value_counts())

category
物流问题    30
退款问题    30
账户问题    30
商品问题    30
Name: count, dtype: int64


In [27]:
X = tickets['text_clean']
y = tickets['category']

In [28]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f'训练集: {len(X_train)},'
      f'验证集: {len(X_valid)},'
      f'测试集: {len(X_test)},')

训练集: 72,验证集: 24,测试集: 24,


In [29]:
# 为训练集每类补充 5 条文本
extra_templates = {
    "物流问题": [
        "快递一直停在转运中心没有动静",
        "包裹显示已发出但查不到轨迹",
        "物流更新太慢了好几天没变化",
        "快递员没有按约定时间派送",
        "订单显示已签收但我没收到货",
    ],
    "退款问题": [
        "申请退款后一直没有审核通过",
        "退款金额和实际支付不一致",
        "退货寄回后迟迟没有退款",
        "取消订单后钱还没退回来",
        "退款到账时间太久了",
    ],
    "账户问题": [
        "登录时提示账号异常无法进入",
        "修改密码后还是登不上",
        "手机号换了解绑不了旧账号",
        "账号被冻结了怎么解封",
        "收不到登录验证码",
    ],
    "商品问题": [
        "收到的商品有明显破损",
        "商品和页面描述不一致",
        "商品用了两天就坏了",
        "包装里缺少配件",
        "商品颜色发错了",
    ],
}

extra_texts, extra_labels = [], []
for cat, texts in extra_templates.items():
    extra_texts.extend(texts)
    extra_labels.extend([cat] * len(texts))

X_train_aug = pd.concat([X_train, pd.Series(extra_texts)], ignore_index=True)
y_train_aug = pd.concat([y_train, pd.Series(extra_labels)], ignore_index=True)

print(f"\n补充后训练集：{len(X_train_aug)}（原 {len(X_train)} + 补充 {len(extra_texts)}）")
print(f"补充后类别分布：\n{y_train_aug.value_counts()}")


补充后训练集：92（原 72 + 补充 20）
补充后类别分布：
退款问题    23
商品问题    23
账户问题    23
物流问题    23
Name: count, dtype: int64


In [30]:
# 同一套模型配置
def build_model():
    return Pipeline([
        (
            'tfidf',
            TfidfVectorizer(
                analyzer='char',
                ngram_range=(1, 2),

            )
        ),
        (
            'model',
            LogisticRegression(
                max_iter=1000,
                random_state=42

            )
        )
    ])

In [31]:
# 训练两个模型
model_before = build_model()
model_before.fit(X_train, y_train)

model_after = build_model()
model_after.fit(X_train_aug, y_train_aug)

# 只在验证集上对比
def evaluate_on_valid(model):
    y_pred = model.predict(X_valid)
    report = classification_report(
        y_valid,
        y_pred,
        output_dict=True,
        digits=3,
        zero_division=0
    )
    return report

res_before = evaluate_on_valid(model_before)
res_after = evaluate_on_valid(model_after)

print(res_before)
print(res_after)

{'商品问题': {'precision': 0.75, 'recall': 1.0, 'f1-score': 0.8571428571428571, 'support': 6.0}, '物流问题': {'precision': 1.0, 'recall': 0.8333333333333334, 'f1-score': 0.9090909090909091, 'support': 6.0}, '账户问题': {'precision': 1.0, 'recall': 0.8333333333333334, 'f1-score': 0.9090909090909091, 'support': 6.0}, '退款问题': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 6.0}, 'accuracy': 0.9166666666666666, 'macro avg': {'precision': 0.9375, 'recall': 0.9166666666666667, 'f1-score': 0.9188311688311688, 'support': 24.0}, 'weighted avg': {'precision': 0.9375, 'recall': 0.9166666666666666, 'f1-score': 0.9188311688311687, 'support': 24.0}}
{'商品问题': {'precision': 0.8571428571428571, 'recall': 1.0, 'f1-score': 0.9230769230769231, 'support': 6.0}, '物流问题': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 6.0}, '账户问题': {'precision': 1.0, 'recall': 0.8333333333333334, 'f1-score': 0.9090909090909091, 'support': 6.0}, '退款问题': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'suppo

In [32]:
# 指标对比表
compare = pd.DataFrame([
    {k: v for k, v in res_before.items() if k != 'y_pred'},
    {k: v for k, v in res_after.items() if k != 'y_pred'}
])

print(compare.to_string(index=False))

                                                                                            商品问题                                                                                             物流问题                                                                                             账户问题                                                               退款问题  accuracy                                                                                                       macro avg                                                                                                    weighted avg
              {'precision': 0.75, 'recall': 1.0, 'f1-score': 0.8571428571428571, 'support': 6.0} {'precision': 1.0, 'recall': 0.8333333333333334, 'f1-score': 0.9090909090909091, 'support': 6.0} {'precision': 1.0, 'recall': 0.8333333333333334, 'f1-score': 0.9090909090909091, 'support': 6.0} {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 6.0}  0.916667            {'precision': 0.9375, 'recall

In [33]:
# 错误样本对比

def show_errors(model):
    y_pred = model.predict(X_valid)
    errors = pd.DataFrame({
        'text': X_valid.values,
        'true': y_valid.values,
        'pred': y_pred,
    })
    errors = errors[errors['true'] != errors['pred']]
    if len(errors):
        print(errors.to_string(index=False))
    else:
        print("无错误样本")
    return errors

err_before = show_errors(model_before)
err_after = show_errors(model_after)
print()
print("验证集before错误样本：")
print(len(err_before))
print("验证集after错误样本：")
print(len(err_after))

      text true pred
同城配送超过预计时间 物流问题 商品问题
 绑定邮箱已经停用了 账户问题 商品问题
     text true pred
绑定邮箱已经停用了 账户问题 商品问题

验证集before错误样本：
2
验证集after错误样本：
1


3. **可选挑战**：增加“发票问题”类别，分别准备不少于 15 条训练文本、5 条验证文本和 5 条测试文本，并说明它与其他类别的判断边界。

In [34]:
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loader(features, labels, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long)
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
    )


In [35]:
def train_model(model, train_loader, loss_function, optimizer, epochs):
    average_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad()
            logits = model(batch_features)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        average_losses.append(total_loss / len(train_loader))

    return average_losses

In [36]:
def compute_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn ) > 0 else 0
    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0
    )
    return {
        'accuracy': round(accuracy, 3),
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3),
    }

In [37]:
from sklearn.metrics import precision_score, recall_score
def select_threshold(probabilities, y_true, thresholds, min_recall):
    best_threshold = None
    best_precision = -1
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        recall = recall_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0
        )
        if recall < min_recall:
            continue
        precision = precision_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0
        )
        if precision > best_precision:
            best_threshold = threshold
            best_precision = precision

    return best_threshold